In [44]:
from pathlib import Path
import socket
import pandas as pd
from pyfame.pyfame import BoEFAMEReader

# =========================================================
# CONFIG
# =========================================================

PROJECT_ROOT = Path(r"C:\Users\344792\Gokce\GIT PROJECTS\DisaggCPI\CPI-disaggregation-in-PT")
OUTPUT_DIR = PROJECT_ROOT / "data" / "fame_exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FAME_HOST = "localhost"
FAME_PORT = 12345

reader = BoEFAMEReader()

# =========================================================
# EXACT FORMULAS FROM EXCEL (B10 ROW)
# =========================================================

EXPRESSIONS = {
    "cpi": 'conv2qa(X12(stifidx.m,MULT,1990m2,2025m12))',
    "services": 'conv2qa(X12(stif_srv.m,MULT,1990m2,2025m12))',
    "core_gds": 'conv2qa(X12(stif_gds_xeatfnab.m,MULT,1990m2,2025m12))',
    "food_at": 'conv2qa(X12(cpi_aggregation(stif_fnab.m, booze.m, fags.m,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#),MULT,1990m2,2025m12))',
    "energy": 'conv2qa(X12(stif_pu.m,MULT,1990m2,2025m12))',
    "pmdef": 'pmdef',   # ✅ works fine
    "eer": 'eer',       # ✅ works fine
    "infl_exp": 'BPROJA(conv2qa(cgrp.m),PUB12MEDIAN.Q,"E")'
}

# =========================================================
# HELPERS
# =========================================================

def check_fame():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2)
    if s.connect_ex((FAME_HOST, FAME_PORT)) != 0:
        raise RuntimeError("FAME service not running")
    s.close()


def normalize_dates(df):
    df = df.copy()

    dt = pd.to_datetime(df["date"], errors="coerce")

    # quarter start → quarter end
    qstart = dt.dt.is_quarter_start.fillna(False)
    dt = dt.where(~qstart, dt - pd.Timedelta(days=1))

    df["date"] = dt.dt.to_period("Q").dt.to_timestamp(how="end")
    return df


# ✅ EXACT REPLICATION OF Excel FAMEData(...)
def read_series(expr, name):

    df = reader.read(expr)

    if df is None or df.empty:
        raise ValueError(f"FAME returned empty for: {name}")

    df = df.copy()

    # Handle different pyfame outputs
    if "index" in df.columns:
        df = df.rename(columns={"index": "date", "value": name})
    elif "date" in df.columns:
        value_col = [c for c in df.columns if c != "date"][-1]
        df = df.rename(columns={value_col: name})
    else:
        df = df.iloc[:, :2]
        df.columns = ["date", name]

    df = normalize_dates(df)

    # ✅ THIS IS THE CRITICAL PART (FAMEData window)
    df = df[
        (df["date"] >= pd.Timestamp("1990-04-01")) &
        (df["date"] <= pd.Timestamp("2025-12-31"))
    ].copy()

    df = df.dropna(subset=[name])

    return df[["date", name]].reset_index(drop=True)


# =========================================================
# MAIN
# =========================================================

check_fame()

print("\nBuilding estimation data (Excel replication)...")

panel = None

for name, expr in EXPRESSIONS.items():
    print("  ->", name)

    series = read_series(expr, name)

    if panel is None:
        panel = series
    else:
        panel = panel.merge(series, on="date", how="outer")

panel = panel.sort_values("date").reset_index(drop=True)

print("\n✅ Done (first rows)")
print(panel.head())

# =========================================================
# EXPORT
# =========================================================

panel.to_csv(OUTPUT_DIR / "estimation_data_latest.csv", index=False)
panel.to_parquet(OUTPUT_DIR / "estimation_data_latest.parquet", index=False)

print("\n✅ Files written")


Building estimation data (Excel replication)...
  -> cpi


ValueError: FAME returned empty for: cpi